In [1]:
import pandas as pd
skempi = pd.read_csv('../data/raw/skempi_v2.csv', sep = ';')
skempi.head()

,#Pdb,Mutation(s)_PDB,Mutation(s)_cleaned,iMutation_Location(s),Hold_out_type,Hold_out_proteins,Affinity_mut (M),Affinity_mut_parsed,Affinity_wt (M),Affinity_wt_parsed,...,koff_mut_parsed,koff_wt (s^(-1)),koff_wt_parsed,dH_mut (kcal mol^(-1)),dH_wt (kcal mol^(-1)),dS_mut (cal mol^(-1) K^(-1)),dS_wt (cal mol^(-1) K^(-1)),Notes,Method,SKEMPI version
0,1CSE_E_I,LI45G,LI38G,COR,Pr/PI,Pr/PI,5.26E-11,5.260000e-11,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1
1,1CSE_E_I,LI45S,LI38S,COR,Pr/PI,Pr/PI,8.33E-12,8.330000e-12,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1
2,1CSE_E_I,LI45P,LI38P,COR,Pr/PI,Pr/PI,1.02E-07,1.020000e-07,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1
3,1CSE_E_I,LI45I,LI38I,COR,Pr/PI,Pr/PI,1.72E-10,1.720000e-10,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1
4,1CSE_E_I,LI45D,LI38D,COR,Pr/PI,Pr/PI,1.92E-09,1.920000e-09,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1


In [2]:
import numpy as np
R = 1.987e-3  # kcal/(mol*K)
T = 298       # K
RT = R * T    # ~0.593 kcal/mol

skempi = skempi.dropna(subset=['Affinity_mut_parsed', 'Affinity_wt_parsed'])

skempi['ddG'] = RT * np.log(skempi['Affinity_mut_parsed'] / skempi['Affinity_wt_parsed'])

skempi.head()

,#Pdb,Mutation(s)_PDB,Mutation(s)_cleaned,iMutation_Location(s),Hold_out_type,Hold_out_proteins,Affinity_mut (M),Affinity_mut_parsed,Affinity_wt (M),Affinity_wt_parsed,...,koff_wt (s^(-1)),koff_wt_parsed,dH_mut (kcal mol^(-1)),dH_wt (kcal mol^(-1)),dS_mut (cal mol^(-1) K^(-1)),dS_wt (cal mol^(-1) K^(-1)),Notes,Method,SKEMPI version,ddG
0,1CSE_E_I,LI45G,LI38G,COR,Pr/PI,Pr/PI,5.26E-11,5.260000e-11,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1,2.279322
1,1CSE_E_I,LI45S,LI38S,COR,Pr/PI,Pr/PI,8.33E-12,8.330000e-12,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1,1.188121
2,1CSE_E_I,LI45P,LI38P,COR,Pr/PI,Pr/PI,1.02E-07,1.020000e-07,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1,6.761723
3,1CSE_E_I,LI45I,LI38I,COR,Pr/PI,Pr/PI,1.72E-10,1.720000e-10,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1,2.980860
4,1CSE_E_I,LI45D,LI38D,COR,Pr/PI,Pr/PI,1.92E-09,1.920000e-09,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1,4.409415


In [3]:
skempi.to_csv('processed_skempi.csv')

In [4]:
from Bio.PDB import PDBParser, PPBuilder

parser = PDBParser(QUIET=True)

ppb = PPBuilder()
def get_chain(pdb_file, chain_id):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)

    ppb = PPBuilder()
    
    chain_sequence = ""
    
    for model in structure:
        for chain in model:
            if chain.id == chain_id:
                polypeptides = ppb.build_peptides(chain)
                for poly in polypeptides:
                    chain_sequence += str(poly.get_sequence())
    return chain_sequence

In [5]:
import pandas as pd
from tqdm import tqdm
from statistics import mean
import warnings
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)

column_use = "Mutation(s)_cleaned"


pdb_cache = {}

def get_chain_cached(pdb_id, chain_id):

    if pdb_id not in pdb_cache:
        pdb_cache[pdb_id] = {}
        pdb_path = f"../../data/raw/PDBs/{pdb_id}.pdb"
        with open(pdb_path) as f:
            pdb_lines = f.readlines()
        # 可复用已有 get_chain 函数
        for line in pdb_lines:
            if line.startswith("ATOM") and line[21] not in pdb_cache[pdb_id]:
                pdb_cache[pdb_id][line[21]] = get_chain(pdb_path, line[21])
    return pdb_cache[pdb_id].get(chain_id, "")

import warnings

warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)
matched_rows = []
unmatched_rows = []

column_use = 'Mutation(s)_cleaned'

for index in tqdm(range(len(skempi))):
    sample = skempi.iloc[index]
    
    if len(sample[column_use].split(',')) > 1:
        continue
    
    pdb_id = sample['#Pdb'].split('_')[0]
    chain = get_chain(f"../data/raw/PDBs/{pdb_id}.pdb", sample[column_use][1])
    
    original_aa = sample[column_use][0]
    position = int(sample[column_use][2:-1]) - 1
    mutation_aa = sample[column_use][-1]

    if position > len(chain) - 1:
        unmatched_rows.append(sample)
        continue
        
    pdb_aa = chain[position]
    
    if pdb_aa == original_aa:
        sample['mut0'] = chain
        sample['mut1'] = chain[:position] + mutation_aa + chain[1 + position:]
        chains = sample['#Pdb'].split('_')[1:]
        
        another = chains[1] if sample[column_use][1] == chains[0] else chains[0]
        sample['par0'] = get_chain(f"../data/raw/PDBs/{pdb_id}.pdb", another)
        matched_rows.append(sample)
    else:
        unmatched_rows.append(sample)

matched_df = pd.DataFrame(matched_rows)
unmatched_df = pd.DataFrame(unmatched_rows)

100%|██████████| 6798/6798 [18:16<00:00,  6.20it/s]


In [5]:
matched_df.head()

,#Pdb,Mutation(s)_PDB,Mutation(s)_cleaned,iMutation_Location(s),Hold_out_type,Hold_out_proteins,Affinity_mut (M),Affinity_mut_parsed,Affinity_wt (M),Affinity_wt_parsed,...,dH_wt (kcal mol^(-1)),dS_mut (cal mol^(-1) K^(-1)),dS_wt (cal mol^(-1) K^(-1)),Notes,Method,SKEMPI version,ddG,mut0,mut1,par0
0,1CSE_E_I,LI45G,LI38G,COR,Pr/PI,Pr/PI,5.26E-11,5.260000e-11,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,IASP,1,2.279322,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTLDLRYNRVR...,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTGDLRYNRVR...,AQTVPYGIPLIKADKVQAQGFKGANVKVAVLDTGIQASHPDLNVVG...
1,1CSE_E_I,LI45S,LI38S,COR,Pr/PI,Pr/PI,8.33E-12,8.330000e-12,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,IASP,1,1.188121,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTLDLRYNRVR...,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTSDLRYNRVR...,AQTVPYGIPLIKADKVQAQGFKGANVKVAVLDTGIQASHPDLNVVG...
2,1CSE_E_I,LI45P,LI38P,COR,Pr/PI,Pr/PI,1.02E-07,1.020000e-07,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,IASP,1,6.761723,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTLDLRYNRVR...,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTPDLRYNRVR...,AQTVPYGIPLIKADKVQAQGFKGANVKVAVLDTGIQASHPDLNVVG...
3,1CSE_E_I,LI45I,LI38I,COR,Pr/PI,Pr/PI,1.72E-10,1.720000e-10,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,IASP,1,2.980860,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTLDLRYNRVR...,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTIDLRYNRVR...,AQTVPYGIPLIKADKVQAQGFKGANVKVAVLDTGIQASHPDLNVVG...
4,1CSE_E_I,LI45D,LI38D,COR,Pr/PI,Pr/PI,1.92E-09,1.920000e-09,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,IASP,1,4.409415,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTLDLRYNRVR...,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTDDLRYNRVR...,AQTVPYGIPLIKADKVQAQGFKGANVKVAVLDTGIQASHPDLNVVG...


In [6]:
positions = []
for index in tqdm(range(len(matched_df))):
    sample = matched_df.iloc[index]
    
    position = int(sample[column_use][2:-1]) - 1

    positions.append([(position, position + 1)])

#positions_mut0
matched_df['positions_mut0'] = positions
matched_df['positions_mut1'] = positions
matched_df.head()

100%|██████████| 4920/4920 [00:00<00:00, 7835.64it/s]


,#Pdb,Mutation(s)_PDB,Mutation(s)_cleaned,iMutation_Location(s),Hold_out_type,Hold_out_proteins,Affinity_mut (M),Affinity_mut_parsed,Affinity_wt (M),Affinity_wt_parsed,...,dS_wt (cal mol^(-1) K^(-1)),Notes,Method,SKEMPI version,ddG,mut0,mut1,par0,positions_mut0,positions_mut1
0,1CSE_E_I,LI45G,LI38G,COR,Pr/PI,Pr/PI,5.26E-11,5.260000e-11,1.12E-12,1.120000e-12,...,NaN,NaN,IASP,1,2.279322,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTLDLRYNRVR...,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTGDLRYNRVR...,AQTVPYGIPLIKADKVQAQGFKGANVKVAVLDTGIQASHPDLNVVG...,"[(37, 38)]","[(37, 38)]"
1,1CSE_E_I,LI45S,LI38S,COR,Pr/PI,Pr/PI,8.33E-12,8.330000e-12,1.12E-12,1.120000e-12,...,NaN,NaN,IASP,1,1.188121,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTLDLRYNRVR...,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTSDLRYNRVR...,AQTVPYGIPLIKADKVQAQGFKGANVKVAVLDTGIQASHPDLNVVG...,"[(37, 38)]","[(37, 38)]"
2,1CSE_E_I,LI45P,LI38P,COR,Pr/PI,Pr/PI,1.02E-07,1.020000e-07,1.12E-12,1.120000e-12,...,NaN,NaN,IASP,1,6.761723,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTLDLRYNRVR...,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTPDLRYNRVR...,AQTVPYGIPLIKADKVQAQGFKGANVKVAVLDTGIQASHPDLNVVG...,"[(37, 38)]","[(37, 38)]"
3,1CSE_E_I,LI45I,LI38I,COR,Pr/PI,Pr/PI,1.72E-10,1.720000e-10,1.12E-12,1.120000e-12,...,NaN,NaN,IASP,1,2.980860,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTLDLRYNRVR...,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTIDLRYNRVR...,AQTVPYGIPLIKADKVQAQGFKGANVKVAVLDTGIQASHPDLNVVG...,"[(37, 38)]","[(37, 38)]"
4,1CSE_E_I,LI45D,LI38D,COR,Pr/PI,Pr/PI,1.92E-09,1.920000e-09,1.12E-12,1.120000e-12,...,NaN,NaN,IASP,1,4.409415,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTLDLRYNRVR...,KSFPEVVGKTVDQAREYFTLHYPQYNVYFLPEGSPVTDDLRYNRVR...,AQTVPYGIPLIKADKVQAQGFKGANVKVAVLDTGIQASHPDLNVVG...,"[(37, 38)]","[(37, 38)]"


In [7]:
data = matched_df
data['id'] = range(1, len(data) + 1)
data.to_pickle('data_skempi.dataset')

In [8]:
#!source /usr/local/Ascend/ascend-toolkit/set_env.sh
import torch

import torch.nn as nn
import math
from functools import partial
from esm.modules import ContactPredictionHead, ESM1bLayerNorm, RobertaLMHead, TransformerLayer, MultiheadAttention 
import esm
from typing import Union
class ESMModelWrapper(nn.Module):
    def __init__(self, model):
        super(ESMModelWrapper, self).__init__()
        self.model = model

    def forward(self, batch_tokens, repr_layers=[33], return_contacts=False):
        return self.model(batch_tokens, repr_layers=repr_layers, return_contacts=return_contacts)


class ESMFeatureEncoder(nn.Module):
    def __init__(self):
        super(ESMFeatureEncoder, self).__init__()
        self.device = 'cuda:1'
        self.model, self.alphabet = esm.pretrained.esm2_t33_650M_UR50D()
        self.model.to(self.device)
        # self.model = torch.nn.DataParallel(self.model, device_ids=[0, 1, 2])
        self.model.eval()  # Set the model to evaluation mode
        self.batch_converter = self.alphabet.get_batch_converter()
        self.padding_idx = self.alphabet.padding_idx
        # Wrap the model with the ESMModelWrapper
        self.model = ESMModelWrapper(self.model)

    def encode(self, sequences):
        batch_labels, batch_strs, batch_tokens = self.batch_converter([(str(0), sequence) for sequence in sequences])
        batch_tokens = batch_tokens.to(self.device)
        batch_mask = batch_tokens.eq(self.padding_idx)
        # print(batch_tokens.shape)
        with torch.no_grad():
            results = self.model(batch_tokens, repr_layers=[33], return_contacts=False)
        token_representations = results['representations'][33]
        # print(results['representations'][33].mean(dim=1).unsqueeze(1).shape)
        return token_representations,batch_mask

# esm_model = ESMFeatureEncoder()
# a, b = esm_model.encode(['TTT'])

In [20]:
from tqdm import tqdm
import numpy as np
import os
#ESM_feature ---need to run for the first time
esm_model = ESMFeatureEncoder()
for i in tqdm(range(len(data))):
    sample = data.iloc[i]
    file_path = f'../../data/middlefile/skempi_ESM_feature/{sample["id"]}/'
    if os.path.exists(file_path):
        continue
    embedding, mask = esm_model.encode([sample['mut0']])
    mut0 = embedding.detach().cpu().numpy()
    embedding, mask = esm_model.encode([sample['mut1']])
    mut1 = embedding.detach().cpu().numpy()
    embedding, mask = esm_model.encode([sample['par0']])
    par0 = embedding.detach().cpu().numpy()
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    np.save(file_path + 'mut0.npy', mut0)
    np.save(file_path + 'mut1.npy', mut1)
    np.save(file_path + 'par0.npy', par0)

100%|██████████| 4920/4920 [31:59<00:00,  2.56it/s]


In [9]:
import numpy as np
def positional_encoding(max_len, d_model):

    position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))
    
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    pe.requires_grad = False
    return pe

len_range = 30
position_embedding = positional_encoding(2 * (2 * len_range + 1) + 2 , 1280).numpy()
position_embedding[2 * len_range + 3:, :].shape, position_embedding.shape

((61, 1280), (124, 1280))

In [10]:
mut0_f_list, mut1_f_list, par0_f_list = [], [], []
for i in tqdm(range(len(data))):
    sample = data.iloc[i]
    correct = True
    file_path = f'../../data/middlefile/skempi_ESM_feature/{sample["id"]}/'
    mut0_feature = np.load(file_path + 'mut0.npy', allow_pickle=True).squeeze(0)

    mut1_feature = np.load(file_path + 'mut1.npy', allow_pickle=True).squeeze(0)

    par0_feature = np.load(file_path + 'par0.npy', allow_pickle=True).squeeze(0)

    parts_mut0 = []
    parts_mut1 = []
    parts_mut0.append(np.expand_dims(mut0_feature.mean(axis=0), axis = 0) + position_embedding[:1, :])
    parts_mut1.append(np.expand_dims(mut1_feature.mean(axis=0), axis = 0) + position_embedding[2 * len_range + 2 : 2 * len_range + 3, :])
    for j in range(len(sample['positions_mut0'])):
        # if sample['mut1'][sample['positions_mut1'][j][0] : sample['positions_mut1'][j][1]] != sample['Resulting sequence'][j]
        
        center = int((sample['positions_mut0'][j][0] + sample['positions_mut0'][j][1]) / 2)

        # start = max(center - len_range, 0)
        start = max(center - len_range + 1, 0)
        part_mut0 = mut0_feature[start: center + len_range + 1 + 1, :]
        if start == 0:
            positions_ed = position_embedding[1 : 2 * len_range + 1 + 1, : ][-part_mut0.shape[0] : , :]
        else:
            positions_ed = position_embedding[1 : 2 * len_range + 1 + 1, : ][ : part_mut0.shape[0] , :]
        parts_mut0.append(part_mut0 + positions_ed)


        center = int((sample['positions_mut1'][j][0] + sample['positions_mut1'][j][1]) / 2)

        # start = max(center - len_range, 0)
        start = max(center - len_range + 1, 0)
        part_mut1 = mut1_feature[start: center + len_range + 1 + 1, :]
        if start == 0:
            positions_ed = position_embedding[3 + 2 * len_range : , : ][-part_mut1.shape[0] : , :]
        else:
            positions_ed = position_embedding[3 + 2 * len_range : , : ][ : part_mut1.shape[0] , :]
        parts_mut1.append(part_mut1 + positions_ed)

    # result = np.concatenate(all_arrays, axis=0)
    mut0_f_list.append(np.concatenate(parts_mut0, axis=0))
    mut1_f_list.append(np.concatenate(parts_mut1, axis=0))
    par0_f_list.append(par0_feature)
        # if sample['mut0'][start: center + len_range + 1][len_range] != sample['Original sequence'][j]:
        #     correct = False
        #     break
    
data['mut0_f'] = mut0_f_list
data['mut1_f'] = mut1_f_list
data['par0_f'] = par0_f_list

100%|██████████| 4920/4920 [01:23<00:00, 59.04it/s] 


In [11]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

train_df, temp_clusters = train_test_split(
    data, test_size=0.30, random_state=42
)

val_df, test_df = train_test_split(
    temp_clusters, test_size=0.50, random_state=42
)

print(f"train_set: {len(train_df)} sample")
print(f"test_set: {len(val_df)} sample")
print(f"validation_set: {len(test_df)} sample")

train_set: 3444 sample
test_set: 738 sample
validation_set: 738 sample


In [12]:
# from torch.utils.data import DataLoader, Dataset
# from tqdm import tqdm
# device = 'cuda:1'
# epoch = 30
# accum_steps = 10
# batch_size = 8

# import torch
# from torch.utils.data import Dataset, DataLoader
# import numpy as np
# from tqdm import tqdm
# import torch.nn.functional as F
# import json
# from model_ddg import TAPPI

# class TAPPI_Dataset(Dataset):
#     def __init__(self, df):
#         self.df = df

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         sample = self.df.iloc[idx]

#         return (
#             sample['mut0_f'], 
#             sample['mut1_f'], 
#             sample['par0_f'], 
#             sample['ddG']
#         )


# def tappi_collate_fn(batch):


#     def pad_features(feature_list):
#         max_len = max(f.shape[0] for f in feature_list)
#         dim = feature_list[0].shape[1]
#         padded, mask = [], []

#         for f in feature_list:
#             pad_len = max_len - f.shape[0]
#             padded_f = np.pad(f, ((0, pad_len), (0, 0)), mode='constant', constant_values=0)
#             padded.append(padded_f)

#             m = np.concatenate([np.zeros(f.shape[0]), np.ones(pad_len)]).astype(bool)
#             mask.append(m)

#         padded_tensor = torch.tensor(np.stack(padded), dtype=torch.float32)
#         mask_tensor = torch.tensor(np.stack(mask), dtype=torch.bool)
#         return padded_tensor, mask_tensor
#     mut0_list, mut1_list, par0_list, labels = zip(*batch)

#     # padding & mask
#     mut0, mut0_mask = pad_features(mut0_list)
#     mut1, mut1_mask = pad_features(mut1_list)
#     par0, par0_mask = pad_features(par0_list)

#     labels = torch.tensor(labels, dtype=torch.long)

#     return mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels

# # train_dataset = TAPPI_Dataset(train_df)
# # val_dataset = TAPPI_Dataset(val_df)
# # train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# # val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# train_dataset = TAPPI_Dataset(train_df)
# val_dataset = TAPPI_Dataset(val_df)
# num_workers = 8
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=batch_size,
#     shuffle=True,
#     collate_fn=tappi_collate_fn,
#     num_workers=num_workers 
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=batch_size,
#     shuffle=False,
#     collate_fn=tappi_collate_fn,
#     num_workers=num_workers
# )

# model = TAPPI(num_layers = 20).to(device)


# optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

# loss_train_all = []
# loss_val_all = []

# def compute_accuracy(logits, labels):
#     preds = torch.argmax(logits, dim=1)
#     correct = (preds == labels).sum().item()
#     return correct / len(labels)



# pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {0+1}/{epoch}")
# for step, (mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels) in pbar:
#     mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels = (
#         mut0.to(device),
#         mut1.to(device),
#         par0.to(device),
#         mut0_mask.to(device),
#         mut1_mask.to(device),
#         par0_mask.to(device),
#         labels.to(device).unsqueeze(1),
#     )
#     result = model(mut0, mut1, par0, torch.cat([mut0_mask, mut1_mask], dim=1), par0_mask)
#     var = labels - result
#     break


In [13]:
labels.shape

NameError: name 'labels' is not defined

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
from tqdm import tqdm
from torch.utils.data import DataLoader
from model_ddg import TAPPI
from torch.utils.data import Dataset

device = 'cuda:1'
epoch = 300
accum_steps = 10
batch_size = 8

def compute_mse(preds, labels):
    return ((preds - labels) ** 2).mean().item()

class TAPPI_Dataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample = self.df.iloc[idx]
        return (
            sample['mut0_f'], 
            sample['mut1_f'], 
            sample['par0_f'], 
            sample['ddG']
        )

def tappi_collate_fn(batch):
    def pad_features(feature_list):
        max_len = max(f.shape[0] for f in feature_list)
        dim = feature_list[0].shape[1]
        padded, mask = [], []
        
        for f in feature_list:
            pad_len = max_len - f.shape[0]
            padded_f = np.pad(f, ((0, pad_len), (0, 0)), mode='constant', constant_values=0)
            padded.append(padded_f)
            
            m = np.concatenate([np.zeros(f.shape[0]), np.ones(pad_len)]).astype(bool)
            mask.append(m)
            
        padded_tensor = torch.tensor(np.stack(padded), dtype=torch.float32)
        mask_tensor = torch.tensor(np.stack(mask), dtype=torch.bool)
        return padded_tensor, mask_tensor
    
    mut0_list, mut1_list, par0_list, labels = zip(*batch)

    # padding & mask
    mut0, mut0_mask = pad_features(mut0_list)
    mut1, mut1_mask = pad_features(mut1_list)
    par0, par0_mask = pad_features(par0_list)

    labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)  # 连续值

    return mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels

train_dataset = TAPPI_Dataset(train_df)
val_dataset = TAPPI_Dataset(val_df)
num_workers = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=tappi_collate_fn,
    num_workers=num_workers
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=tappi_collate_fn,
    num_workers=num_workers
)

model = TAPPI(num_layers=39).to(device)
model.load_state_dict(torch.load("../../data/middlefile/75mse.pth", map_location=device))
loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

loss_train_all = []
loss_val_all = []

for ep in range(epoch):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    all_train_preds = []
    all_train_labels = []

    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {ep+1}/{epoch}")
    for step, (mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels) in pbar:
        mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels = (
            mut0.to(device),
            mut1.to(device),
            par0.to(device),
            mut0_mask.to(device),
            mut1_mask.to(device),
            par0_mask.to(device),
            labels.to(device)
        )

        preds = model(mut0, mut1, par0, torch.cat([mut0_mask, mut1_mask], dim=1), par0_mask)

        loss = loss_fn(preds, labels)
        loss = loss / accum_steps
        loss.backward()

        if (step + 1) % accum_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item() * accum_steps

        all_train_preds.append(preds.detach().cpu())
        all_train_labels.append(labels.detach().cpu())

        pbar.set_postfix(loss=f"{train_loss/(step+1):.4f}")

    all_train_preds = torch.cat(all_train_preds)
    all_train_labels = torch.cat(all_train_labels)
    train_mse = compute_mse(all_train_preds, all_train_labels)
    train_loss /= len(train_loader)
    preds_np = all_val_preds.numpy().flatten()
    labels_np = all_val_labels.numpy().flatten()
    pearson_r, p_value = pearsonr(preds_np, labels_np)
    loss_train_all.append(pearson_r)
    with open("../../data/middlefile/pearson_train_all.json", "w") as f:
        json.dump(loss_train_all, f)

    model.eval()
    val_loss = 0.0
    all_val_preds = []
    all_val_labels = []

    with torch.no_grad():
        for mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels in val_loader:
            mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels = (
                mut0.to(device),
                mut1.to(device),
                par0.to(device),
                mut0_mask.to(device),
                mut1_mask.to(device),
                par0_mask.to(device),
                labels.to(device)
            )

            preds = model(mut0, mut1, par0, torch.cat([mut0_mask, mut1_mask], dim=1), par0_mask)
            loss = loss_fn(preds, labels)
            val_loss += loss.item()

            all_val_preds.append(preds.detach().cpu())
            all_val_labels.append(labels.detach().cpu())

    all_val_preds = torch.cat(all_val_preds)
    all_val_labels = torch.cat(all_val_labels)
    val_mse = compute_mse(all_val_preds, all_val_labels)

    np.save('../../data/middlefile/ddG_preds.npy', all_val_preds.numpy())
    np.save('../../data/middlefile/ddG_labels.npy', all_val_labels.numpy())
    preds_np = all_val_preds.numpy().flatten()
    labels_np = all_val_labels.numpy().flatten()
    pearson_r, p_value = pearsonr(preds_np, labels_np)
    val_loss /= len(val_loader)
    loss_val_all.append(pearson_r)
    with open("../../data/middlefile/pearson_val_all.json", "w") as f:
        json.dump(loss_val_all, f)

In [39]:
loss_val_all

[2.7835874557495117,
 2.6110658645629883,
 2.584826707839966,
 2.4896328449249268,
 2.416799783706665,
 2.493699550628662,
 2.2693607807159424,
 2.2460429668426514,
 2.3717448711395264,
 2.3015902042388916,
 2.469346761703491,
 2.2223117351531982,
 2.1766111850738525,
 2.1795692443847656,
 2.314420461654663,
 2.498356580734253,
 2.48927903175354,
 2.2020657062530518,
 2.171337842941284,
 2.1373863220214844,
 2.272643804550171,
 2.1637279987335205,
 2.3475441932678223,
 2.3405025005340576,
 2.214588165283203,
 2.3670787811279297,
 2.1091063022613525,
 2.1481211185455322,
 2.1173460483551025,
 2.268301010131836]

In [41]:
import numpy as np
from scipy.stats import pearsonr
preds_np = all_val_preds.numpy().flatten()
labels_np = all_val_labels.numpy().flatten()
pearson_r, p_value = pearsonr(preds_np, labels_np)
pearson_r

0.43504319083364607

In [42]:
for ep in range(200):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    all_train_preds = []
    all_train_labels = []

    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {ep+1}/{epoch}")
    for step, (mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels) in pbar:
        mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels = (
            mut0.to(device),
            mut1.to(device),
            par0.to(device),
            mut0_mask.to(device),
            mut1_mask.to(device),
            par0_mask.to(device),
            labels.to(device)
        )

        preds = model(mut0, mut1, par0, torch.cat([mut0_mask, mut1_mask], dim=1), par0_mask)

        loss = loss_fn(preds, labels)
        loss = loss / accum_steps
        loss.backward()

        if (step + 1) % accum_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item() * accum_steps

        all_train_preds.append(preds.detach().cpu())
        all_train_labels.append(labels.detach().cpu())

        pbar.set_postfix(loss=f"{train_loss/(step+1):.4f}")

    all_train_preds = torch.cat(all_train_preds)
    all_train_labels = torch.cat(all_train_labels)
    train_mse = compute_mse(all_train_preds, all_train_labels)
    train_loss /= len(train_loader)
    preds_np = all_train_preds.numpy().flatten()
    labels_np = all_train_labels.numpy().flatten()
    pearson_r, p_value = pearsonr(preds_np, labels_np)
    loss_train_all.append(pearson_r)
    with open("../../data/middlefile/pearson_train_all.json", "w") as f:
        json.dump(loss_train_all, f)

    model.eval()
    val_loss = 0.0
    all_val_preds = []
    all_val_labels = []

    with torch.no_grad():
        for mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels in val_loader:
            mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels = (
                mut0.to(device),
                mut1.to(device),
                par0.to(device),
                mut0_mask.to(device),
                mut1_mask.to(device),
                par0_mask.to(device),
                labels.to(device)
            )

            preds = model(mut0, mut1, par0, torch.cat([mut0_mask, mut1_mask], dim=1), par0_mask)
            loss = loss_fn(preds, labels)
            val_loss += loss.item()

            all_val_preds.append(preds.detach().cpu())
            all_val_labels.append(labels.detach().cpu())

    all_val_preds = torch.cat(all_val_preds)
    all_val_labels = torch.cat(all_val_labels)
    val_mse = compute_mse(all_val_preds, all_val_labels)

    np.save('../../data/middlefile/ddG_preds.npy', all_val_preds.numpy())
    np.save('../../data/middlefile/ddG_labels.npy', all_val_labels.numpy())
    preds_np = all_val_preds.numpy().flatten()
    labels_np = all_val_labels.numpy().flatten()
    pearson_r, p_value = pearsonr(preds_np, labels_np)
    val_loss /= len(val_loader)
    loss_val_all.append(pearson_r)
    with open("../../data/middlefile/pearson_val_all.json", "w") as f:
        json.dump(loss_val_all, f)

Epoch 10/30:  82%|████████▏ | 352/431 [01:15<00:19,  4.15it/s, loss=1.7410]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Epoch 65/30:  99%|█████████▉| 427/431 [02:35<00:01,  2.73it/s, loss=0.6941]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Epoch 111/30:  97%|█████████▋| 420/431 [01:28<00:02,  4.70it/s, loss=0.3688]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To chang

In [ ]:
import numpy as np
labels = np.concatenate(label_def)
preds = np.concatenate(pred_def)


pred_classes = np.round(preds)
acc = np.mean(pred_classes == labels)

print(f"Accuracy: {acc:.4f}")

In [ ]:
for ep in range(200):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    all_train_preds = []
    all_train_labels = []

    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {ep+1}/{epoch}")
    for step, (mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels) in pbar:
        mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels = (
            mut0.to(device),
            mut1.to(device),
            par0.to(device),
            mut0_mask.to(device),
            mut1_mask.to(device),
            par0_mask.to(device),
            labels.to(device)
        )

        preds = model(mut0, mut1, par0, torch.cat([mut0_mask, mut1_mask], dim=1), par0_mask)

        loss = loss_fn(preds, labels)
        loss = loss / accum_steps
        loss.backward()

        if (step + 1) % accum_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item() * accum_steps

        all_train_preds.append(preds.detach().cpu())
        all_train_labels.append(labels.detach().cpu())

        pbar.set_postfix(loss=f"{train_loss/(step+1):.4f}")

    all_train_preds = torch.cat(all_train_preds)
    all_train_labels = torch.cat(all_train_labels)
    train_mse = compute_mse(all_train_preds, all_train_labels)
    train_loss /= len(train_loader)
    preds_np = all_val_preds.numpy().flatten()
    labels_np = all_val_labels.numpy().flatten()
    pearson_r, p_value = pearsonr(preds_np, labels_np)
    loss_train_all.append(pearson_r)
    with open("../../data/middlefile/pearson_train_all.json", "w") as f:
        json.dump(loss_train_all, f)

    model.eval()
    val_loss = 0.0
    all_val_preds = []
    all_val_labels = []

    with torch.no_grad():
        for mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels in val_loader:
            mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels = (
                mut0.to(device),
                mut1.to(device),
                par0.to(device),
                mut0_mask.to(device),
                mut1_mask.to(device),
                par0_mask.to(device),
                labels.to(device)
            )

            preds = model(mut0, mut1, par0, torch.cat([mut0_mask, mut1_mask], dim=1), par0_mask)
            loss = loss_fn(preds, labels)
            val_loss += loss.item()

            all_val_preds.append(preds.detach().cpu())
            all_val_labels.append(labels.detach().cpu())

    all_val_preds = torch.cat(all_val_preds)
    all_val_labels = torch.cat(all_val_labels)
    val_mse = compute_mse(all_val_preds, all_val_labels)

    np.save('../../data/middlefile/ddG_preds.npy', all_val_preds.numpy())
    np.save('../../data/middlefile/ddG_labels.npy', all_val_labels.numpy())
    preds_np = all_val_preds.numpy().flatten()
    labels_np = all_val_labels.numpy().flatten()
    pearson_r, p_value = pearsonr(preds_np, labels_np)
    val_loss /= len(val_loader)
    loss_val_all.append(pearson_r)
    with open("../../data/middlefile/pearson_val_all.json", "w") as f:
        json.dump(loss_val_all, f)

Epoch 48/30:  34%|███▍      | 147/431 [00:31<01:05,  4.37it/s, loss=0.0927]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Epoch 83/30:  57%|█████▋    | 246/431 [00:53<00:37,  4.96it/s, loss=0.1041]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Epoch 86/30:  14%|█▍        | 61/431 [00:13<01:22,  4.49it/s, loss=0.1375]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change 

In [52]:
torch.save(model.state_dict(), f"../../data/middlefile/75mse.pth")